# Практическая работа №4. Вариант 52

**Студент:** Якупов Амир  
**Тема:** интеллектуальная система сортировки отходов  
**Задача:** классификация отходов по изображениям  
**Датасет:** [TrashNet](https://github.com/garythung/trashnet)  
**Модель:** CNN — `Conv2D → MaxPooling → Conv2D → MaxPooling → Flatten → Dense`

Цель — построить и сравнить несколько конфигураций CNN, выбрать лучшую по отложенной тестовой выборке и проанализировать ошибки.

[Репозиторий проекта](https://github.com/24251205-crypto/pr4-trashnet-cnn-variant-52)

## Полученный результат

При фиксированном разбиении `70% / 15% / 15%` лучшей стала настроенная CNN с Batch Normalization, Dropout 0,20, AdamW и `learning_rate=5e-4`.

| Эксперимент | Accuracy | Macro-F1 |
|---|---:|---:|
| Базовая CNN | 0,621 | 0,598 |
| Сильная регуляризация | 0,608 | 0,585 |
| **Настроенная CNN** | **0,679** | **0,657** |

Ниже приведён полностью воспроизводимый код. При повторном запуске возможны небольшие отличия из-за особенностей параллельных вычислений.

In [ ]:
# Основные библиотеки (в Google Colab PyTorch уже установлен)
import copy, json, os, random, time, urllib.request, zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", DEVICE)

## 1. Загрузка датасета

In [ ]:
DATA_URL = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
ARCHIVE = Path("/content/dataset-resized.zip")
DATA_ROOT = Path("/content/dataset-resized")

if not DATA_ROOT.exists():
    urllib.request.urlretrieve(DATA_URL, ARCHIVE)
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall("/content")

CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
CLASS_RU = {
    "cardboard": "картон", "glass": "стекло", "metal": "металл",
    "paper": "бумага", "plastic": "пластик", "trash": "прочий мусор"
}

rows = []
for label in CLASS_NAMES:
    for path in sorted((DATA_ROOT / label).glob("*.jpg")):
        rows.append({"path": str(path), "relative_path": path.relative_to(DATA_ROOT).as_posix(), "label": label})

print("Всего изображений:", len(rows))
pd.Series([r["label"] for r in rows]).value_counts().reindex(CLASS_NAMES)

Официальный TrashNet содержит 2527 цветных изображений шести классов: картон, стекло, металл, бумага, пластик и прочий мусор. Класс `trash` заметно меньше остальных, поэтому кроме accuracy используется Macro-F1.

In [ ]:
labels = [r["label"] for r in rows]
train_rows, temp_rows = train_test_split(rows, test_size=0.30, stratify=labels, random_state=SEED)
temp_labels = [r["label"] for r in temp_rows]
val_rows, test_rows = train_test_split(temp_rows, test_size=0.50, stratify=temp_labels, random_state=SEED)

manifest = []
for split_name, split_rows in [("train", train_rows), ("validation", val_rows), ("test", test_rows)]:
    for row in split_rows:
        manifest.append({"relative_path": row["relative_path"], "label": row["label"], "label_ru": CLASS_RU[row["label"]], "split": split_name})
pd.DataFrame(manifest).sort_values("relative_path").to_csv("/content/trashnet_manifest.csv", index=False)
print({"train": len(train_rows), "validation": len(val_rows), "test": len(test_rows)})

In [ ]:
counts = pd.Series([r["label"] for r in rows]).value_counts().reindex(CLASS_NAMES)
ax = counts.rename(index=CLASS_RU).plot(kind="bar", color="#2563eb", figsize=(8, 4))
ax.set(title="Распределение классов TrashNet", xlabel="Класс", ylabel="Количество")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 2. Предобработка

- все изображения переводятся в RGB и масштабируются до `64×64`;
- пиксели преобразуются в тензор и нормализуются средними и стандартными отклонениями ImageNet;
- во втором эксперименте проверяются поворот, отражение и изменение яркости/контраста;
- разбиение стратифицировано и одинаково для всех экспериментов.

In [ ]:
IMAGE_SIZE = 64

def make_transform(augment=False):
    steps = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if augment:
        steps += [
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomRotation(12),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
        ]
    steps += [
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
    return transforms.Compose(steps)

class TrashDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform
        self.label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as image:
            image = self.transform(image.convert("RGB"))
        return image, self.label_to_idx[row["label"]]

## 3. Архитектура CNN

| Слой | Выход для `64×64` | Назначение |
|---|---|---|
| Conv2D, 32 фильтра, 3×3 + ReLU | 32×64×64 | локальные признаки |
| MaxPooling 2×2 | 32×32×32 | уменьшение карты признаков |
| Conv2D, 64 фильтра, 3×3 + ReLU | 64×32×32 | признаки более высокого уровня |
| MaxPooling 2×2 | 64×16×16 | уменьшение размерности |
| Flatten + Dense 128 + Dropout | 128 | классификационные признаки |
| Dense 6 | 6 | логиты шести классов |

Batch Normalization включается только в экспериментах 2 и 3.

In [ ]:
class TrashNetCNN(nn.Module):
    def __init__(self, dropout=0.0, batch_norm=False):
        super().__init__()
        def block(cin, cout):
            layers = [nn.Conv2d(cin, cout, 3, padding=1)]
            if batch_norm:
                layers.append(nn.BatchNorm2d(cout))
            layers += [nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            return layers
        self.features = nn.Sequential(*block(3, 32), *block(32, 64))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 6),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = TrashNetCNN(dropout=0.20, batch_norm=True)
print(model)
print("Параметров:", sum(p.numel() for p in model.parameters()))

## 4. Обучение и оценка

In [ ]:
def one_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, y_true, y_pred = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        y_true += labels.cpu().tolist()
        y_pred += logits.argmax(1).detach().cpu().tolist()
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

@torch.no_grad()
def predict(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for images, labels in loader:
        logits = model(images.to(DEVICE))
        y_true += labels.tolist()
        y_pred += logits.argmax(1).cpu().tolist()
    return y_true, y_pred

def run_experiment(name, epochs, lr, dropout, augment, batch_norm, class_weights, weight_decay):
    train_ds = TrashDataset(train_rows, make_transform(augment))
    val_ds = TrashDataset(val_rows, make_transform(False))
    test_ds = TrashDataset(test_rows, make_transform(False))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

    model = TrashNetCNN(dropout=dropout, batch_norm=batch_norm).to(DEVICE)
    weights = None
    if class_weights:
        counts = Counter(r["label"] for r in train_rows)
        weights = torch.tensor([len(train_rows)/(6*counts[c]) for c in CLASS_NAMES], dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history, best_state, best_loss, stale = [], copy.deepcopy(model.state_dict()), float("inf"), 0

    for epoch in range(1, epochs + 1):
        train_m = one_epoch(model, train_loader, criterion, optimizer)
        val_m = one_epoch(model, val_loader, criterion)
        history.append({"epoch": epoch, "train": train_m, "val": val_m})
        print(name, epoch, "train acc", round(train_m["accuracy"], 3), "val acc", round(val_m["accuracy"], 3))
        if val_m["loss"] < best_loss - 1e-4:
            best_loss, stale, best_state = val_m["loss"], 0, copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= 4:
            break

    model.load_state_dict(best_state)
    y_true, y_pred = predict(model, test_loader)
    return model, history, {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true, "y_pred": y_pred,
    }

In [ ]:
configs = [
    dict(name="baseline", epochs=8, lr=1e-3, dropout=0.0, augment=False, batch_norm=False, class_weights=False, weight_decay=0.0),
    dict(name="strong_regularization", epochs=12, lr=5e-4, dropout=0.35, augment=True, batch_norm=True, class_weights=True, weight_decay=1e-4),
    dict(name="tuned", epochs=14, lr=5e-4, dropout=0.20, augment=False, batch_norm=True, class_weights=False, weight_decay=1e-4),
]

models, histories, results = {}, {}, {}
for cfg in configs:
    name = cfg["name"]
    models[name], histories[name], results[name] = run_experiment(**cfg)

pd.DataFrame([
    {"experiment": name, "accuracy": values["accuracy"], "macro_f1": values["macro_f1"]}
    for name, values in results.items()
]).sort_values("accuracy", ascending=False)

## 5. Фактические результаты зафиксированного запуска

![Сравнение экспериментов](results/experiment_comparison.png)

![Графики обучения](results/training_curves.png)

Настроенная CNN получила `accuracy = 0,679`, `Macro-Precision = 0,663`, `Macro-Recall = 0,661`, `Macro-F1 = 0,657`. На тестовой выборке 258 из 380 изображений классифицированы верно.

In [ ]:
best = results["tuned"]
print(classification_report(best["y_true"], best["y_pred"], target_names=CLASS_NAMES, zero_division=0))
cm = confusion_matrix(best["y_true"], best["y_pred"])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[CLASS_RU[c] for c in CLASS_NAMES], yticklabels=[CLASS_RU[c] for c in CLASS_NAMES])
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.title("Матрица ошибок настроенной CNN")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

![Матрица ошибок](results/confusion_matrix.png)

![Примеры прогнозов](results/prediction_examples.png)

Бумага распознаётся лучше всего (`F1 = 0,79`). Наиболее сложные классы — пластик и прочий мусор (`F1 = 0,55`): их объектов меньше, а визуальные свойства пересекаются со стеклом и металлом.

## 6. Вывод

Разработана CNN для шести классов TrashNet. Сильная регуляризация одновременно несколькими методами ухудшила итоговые метрики: модель недообучилась. Умеренная конфигурация с Batch Normalization и Dropout 0,20 оказалась лучшей и повысила accuracy с 62,1% до 67,9%.

Для дальнейшего улучшения полезны: больше изображений редких классов, более аккуратная аугментация, transfer learning (например, MobileNetV3 или EfficientNet-B0), подбор learning rate по расписанию и анализ качества исходной разметки.

### Краткие ответы на контрольные вопросы

1. **Типы сетей:** полносвязные сети работают с признаковыми векторами; CNN учитывают пространственную структуру изображения; RNN/LSTM обрабатывают последовательности; трансформеры используют механизм внимания.
2. **Dropout:** случайно отключает часть нейронов при обучении и снижает переобучение, но слишком большое значение приводит к недообучению.
3. **Нормализация:** приводит каналы к сопоставимому масштабу, стабилизирует градиенты и ускоряет оптимизацию.
4. **Улучшение модели:** больше и качественнее данных, умеренная аугментация, подбор архитектуры и гиперпараметров, transfer learning, регуляризация и корректная оценка.
5. **Переобучение:** модель запоминает train-выборку и хуже работает на новых данных. Помогают validation-выборка, early stopping, Dropout, weight decay, аугментация и увеличение датасета.